In [ ]:
using LinearAlgebra
using MatrixFunctions  # 行列の指数関数を使う

# スキュー対称行列（so(3)の元）を生成
function hat(v::AbstractVector{<:Real})
    return [
        0.0    -v[3]   v[2];
        v[3]   0.0    -v[1];
       -v[2]   v[1]   0.0
    ]
end

# 交換子: [A, B] = AB - BA
function commutator(A, B)
    return A * B - B * A
end

# ad_X の作用を表す線形写像（Y ↦ [X, Y]）
function adX_map(X)
    return Y -> commutator(X, Y)
end

# ad(X) を行列として構成（リー環の元に作用する写像を具体的に行列表現に変換）
function ad_matrix(X)
    # 基底: so(3)の標準基底行列
    E1 = hat([1.0, 0.0, 0.0])
    E2 = hat([0.0, 1.0, 0.0])
    E3 = hat([0.0, 0.0, 1.0])
    basis = [E1, E2, E3]
    
    # [X, Ei] を基底展開することで ad_X を行列化
    adX = zeros(3, 3)
    for i in 1:3
        comm = commutator(X, basis[i])
        for j in 1:3
            adX[j, i] = 0.5 * tr(comm' * basis[j])  # 内積として tr(AᵗB) を使用
        end
    end
    return adX
end

# 主コード：具体例で確認
v = [1.0, 2.0, 3.0]           # 任意のベクトル
X = hat(v)                   # so(3) の元（スキュー対称行列）
t = 0.1                      # 小さな時間パラメータ

# Ad(exp(tX)) は:  Y ↦ exp(tX) * Y * exp(-tX)
Y = hat([0.5, -1.0, 0.5])    # 適当な so(3) の元 Y を取る
g = exp(t * X)               # exp(tX)
Ad_exp_tX_Y = g * Y * g'     # Ad(exp(tX))(Y)

# exp(t ad(X))(Y) を計算
adX = ad_matrix(X)           # ad(X) の行列表現（3x3）
Y_vec = [0.5, -1.0, 0.5]     # Y の係数ベクトル（so(3)基底で）
exp_adX = exp(t * adX)       # exp(t ad(X))
exp_adX_Y_vec = exp_adX * Y_vec

# Y_vec を so(3) に戻す
E1, E2, E3 = hat.([[1.0,0,0], [0,1.0,0], [0,0,1.0]])
exp_adX_Y = exp_adX_Y_vec[1]*E1 + exp_adX_Y_vec[2]*E2 + exp_adX_Y_vec[3]*E3

# 結果比較
println("Ad(exp(tX))(Y):")
display(Ad_exp_tX_Y)

println("exp(t ad(X))(Y):")
display(exp_adX_Y)

println("ほぼ一致するか: ", isapprox(Ad_exp_tX_Y, exp_adX_Y; atol=1e-10))


LoadError: ArgumentError: Package numpy not found in current path.
- Run `import Pkg; Pkg.add("numpy")` to install the numpy package.